## 2. Стартовый Baseline


In [1]:
# =============================================================================
# ЭТАП 1. Импорт библиотек
# =============================================================================

import numpy as np
import pandas as pd

import lightgbm as lgb

from lightgbm import LGBMClassifier

from sklearn.model_selection import TimeSeriesSplit

from sklearn.metrics import mean_absolute_error

import warnings

warnings.filterwarnings("ignore")

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# =============================================================================
# ЭТАП 2. Загрузка исходных данных
# =============================================================================

DATA_DIR = "/content/drive/MyDrive/IncomePrediction_Data/01_raw"

train_raw = pd.read_csv(
    f"{DATA_DIR}/train.csv",
    sep=";",
    decimal=",",
    low_memory=False
)

test_raw = pd.read_csv(
    f"{DATA_DIR}/test.csv",
    sep=";",
    decimal=",",
    low_memory=False
)

print(f"Train shape: {train_raw.shape}")
print(f"Test shape : {test_raw.shape}")

Train shape: (76786, 224)
Test shape : (73214, 222)


In [4]:
# =============================================================================
# ЭТАП 3. Универсальный парсер числовых значений
# =============================================================================

def safe_parse_float(x):

    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x == "" or x.lower() in ("nan", "none", "null"):
        return np.nan

    x = x.replace(" ", "")
    x = x.replace(",", ".")

    try:
        return float(x)

    except ValueError:
        return np.nan

In [5]:
# =============================================================================
# ЭТАП 4. Определение категориальных и числовых признаков
# =============================================================================

IGNORE_COLS = [

    "id",

    "dt",

    "target",

    "w"

]

all_cols = [

    c

    for c in train_raw.columns

    if c not in IGNORE_COLS

]

known_cat = [

    "gender",

    "adminarea",

    "city_smart_name",

    "addrref",

    "dp_ewb_last_employment_position",

    "dp_ewb_last_organization",

    "incomeValueCategory",

    "nonresident_flag",

    "client_active_flag",

    "accountsalary_out_flag",

    "blacklist_flag",

    "vert_has_app_ru_tinkoff_investing",

    "vert_has_app_ru_vtb_invest",

    "vert_has_app_ru_cian_main",

    "vert_has_app_ru_raiffeisennews"

]

cat_cols = [

    c

    for c in known_cat

    if c in train_raw.columns

]

num_cols = [

    c

    for c in all_cols

    if c not in cat_cols

]

print(f"Числовых признаков      : {len(num_cols)}")
print(f"Категориальных признаков: {len(cat_cols)}")

Числовых признаков      : 205
Категориальных признаков: 15


In [6]:
# =============================================================================
# ЭТАП 5. Подготовка числовых и категориальных признаков
# =============================================================================

train_num = pd.DataFrame(index=train_raw.index)

test_num = pd.DataFrame(index=test_raw.index)


# -----------------------------------------------------------------------------
# Приведение числовых признаков к float
# -----------------------------------------------------------------------------

for col in num_cols:

    train_num[col] = train_raw[col].apply(safe_parse_float)

    test_num[col] = test_raw[col].apply(safe_parse_float)


# -----------------------------------------------------------------------------
# Заполнение пропусков медианами
# -----------------------------------------------------------------------------

medians = {}

for col in num_cols:

    median = train_num[col].median()

    medians[col] = median

    train_num[col] = train_num[col].fillna(median)

    test_num[col] = test_num[col].fillna(median)


# -----------------------------------------------------------------------------
# Подготовка категориальных признаков
# -----------------------------------------------------------------------------

train_cat = pd.DataFrame(index=train_raw.index)

test_cat = pd.DataFrame(index=test_raw.index)


for col in cat_cols:

    train_cat[col] = train_raw[col].copy()

    test_cat[col] = test_raw[col].copy()

    train_cat[col] = (

        train_cat[col]

        .replace("", "MISSING")

        .fillna("MISSING")

    )

    test_cat[col] = (

        test_cat[col]

        .replace("", "MISSING")

        .fillna("MISSING")

    )


print("Числовые признаки:", train_num.shape)

print("Категориальные признаки:", train_cat.shape)

Числовые признаки: (76786, 205)
Категориальные признаки: (76786, 15)


In [7]:
# =============================================================================
# ЭТАП 6. Создание временных признаков
# =============================================================================

def add_date_features(df, dt_series):

    dt = pd.to_datetime(
        dt_series,
        errors="coerce"
    )

    df["dt_year"] = dt.dt.year

    df["dt_month"] = dt.dt.month

    df["dt_quarter"] = dt.dt.quarter

    df["dt_dayofweek"] = dt.dt.dayofweek

    df["dt_dayofyear"] = dt.dt.dayofyear

    df["dt_week"] = dt.dt.isocalendar().week.astype(int)

    df["dt_is_month_start"] = dt.dt.is_month_start.astype(int)

    df["dt_is_month_end"] = dt.dt.is_month_end.astype(int)

    # Циклическое кодирование месяца

    df["month_sin"] = np.sin(
        2 * np.pi * df["dt_month"] / 12
    )

    df["month_cos"] = np.cos(
        2 * np.pi * df["dt_month"] / 12
    )

    # Циклическое кодирование недели

    df["week_sin"] = np.sin(
        2 * np.pi * df["dt_week"] / 52
    )

    df["week_cos"] = np.cos(
        2 * np.pi * df["dt_week"] / 52
    )

    return df


train_num = add_date_features(
    train_num,
    train_raw["dt"]
)

test_num = add_date_features(
    test_num,
    test_raw["dt"]
)

print(train_num.shape)
print(test_num.shape)

(76786, 217)
(73214, 217)


In [8]:
# =============================================================================
# ЭТАП 7. Формирование обучающих выборок
# =============================================================================

# -----------------------------------------------------------------------------
# Объединяем числовые и категориальные признаки
# -----------------------------------------------------------------------------

X_train = pd.concat(

    [train_num, train_cat],

    axis=1

)

X_test = pd.concat(

    [test_num, test_cat],

    axis=1

)


# -----------------------------------------------------------------------------
# Целевая переменная и веса
# -----------------------------------------------------------------------------

y_train = train_raw["target"].apply(
    safe_parse_float
).astype(float)

w_train = train_raw["w"].apply(
    safe_parse_float
).astype(float)


print(y_train.describe())


# -----------------------------------------------------------------------------
# Приводим категориальные признаки к типу category
# -----------------------------------------------------------------------------

for col in cat_cols:

    all_values = pd.concat(

        [

            X_train[col],

            X_test[col]

        ],

        axis=0

    ).astype(str)

    categories = pd.Categorical(
        all_values
    ).categories

    X_train[col] = pd.Categorical(

        X_train[col].astype(str),

        categories=categories

    )

    X_test[col] = pd.Categorical(

        X_test[col].astype(str),

        categories=categories

    )


print("\nТипы признаков:")

print(X_train.dtypes.value_counts())

print("\nРазмерности:")

print("X_train:", X_train.shape)

print("X_test :", X_test.shape)

count    7.678600e+04
mean     9.264824e+04
std      1.124090e+05
min      2.000000e+04
25%      3.970997e+04
50%      6.275413e+04
75%      1.002017e+05
max      1.500000e+06
Name: target, dtype: float64

Типы признаков:
float64     209
category      7
int32         5
int64         3
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
Name: count, dtype: int64

Размерности:
X_train: (76786, 232)
X_test : (73214, 232)


In [9]:
# =============================================================================
# ЭТАП 8. Анализ корреляций с целевой переменной
# =============================================================================

print("Топ-15 признаков по абсолютной корреляции:")

corr = (

    X_train[num_cols]

    .corrwith(y_train)

    .abs()

    .sort_values(ascending=False)

)

print(corr.head(15))

Топ-15 признаков по абсолютной корреляции:
turn_cur_cr_avg_act_v2                                                                0.601267
turn_cur_db_avg_act_v2                                                                0.599549
turn_cur_cr_sum_v2                                                                    0.593990
turn_cur_cr_avg_v2                                                                    0.593990
turn_cur_db_avg_v2                                                                    0.592674
turn_cur_db_sum_v2                                                                    0.592674
avg_6m_all                                                                            0.526679
avg_cur_db_turn                                                                       0.525508
avg_cur_cr_turn                                                                       0.525431
salary_6to12m_avg                                                                     0.504877
avg_3m_

In [10]:
# =============================================================================
# ЭТАП 9. Параметры моделей
# =============================================================================

# -----------------------------------------------------------------------------
# Параметры LightGBM-регрессоров
# -----------------------------------------------------------------------------

lgb_params = {

    "objective": "mae",

    "metric": "mae",

    "learning_rate": 0.02,

    "num_leaves": 64,

    "min_data_in_leaf": 100,

    "feature_fraction": 0.9,

    "bagging_fraction": 0.8,

    "bagging_freq": 1,

    "lambda_l1": 1.0,

    "lambda_l2": 5.0,

    "verbosity": -1,

    "seed": 42,

    "num_threads": -1

}


# -----------------------------------------------------------------------------
# Параметры классификатора
# -----------------------------------------------------------------------------

clf_params = {

    "objective": "multiclass",

    "num_class": 3,

    "learning_rate": 0.02,

    "n_estimators": 1500,

    "num_leaves": 64,

    "subsample": 0.8,

    "colsample_bytree": 0.9,

    "random_state": 42

}

In [11]:
# =============================================================================
# ЭТАП 9. Параметры ансамбля
# =============================================================================

params_1 = {

    "objective": "mae",

    "metric": "mae",

    "learning_rate": 0.02,

    "num_leaves": 64,

    "min_data_in_leaf": 100,

    "feature_fraction": 0.90,

    "bagging_fraction": 0.80,

    "bagging_freq": 1,

    "lambda_l1": 1,

    "lambda_l2": 5,

    "feature_pre_filter": False,

    "verbosity": -1,

    "seed": 42,

    "num_threads": -1
}

params_2 = {

    "objective": "mae",

    "metric": "mae",

    "learning_rate": 0.02,

    "num_leaves": 128,

    "min_data_in_leaf": 100,

    "feature_fraction": 0.80,

    "bagging_fraction": 0.80,

    "bagging_freq": 1,

    "lambda_l1": 1,

    "lambda_l2": 5,

    "feature_pre_filter": False,

    "verbosity": -1,

    "seed": 123,

    "num_threads": -1
}

params_3 = {

    "objective": "mae",

    "metric": "mae",

    "learning_rate": 0.02,

    "num_leaves": 32,

    "min_data_in_leaf": 100,

    "feature_fraction": 0.70,

    "bagging_fraction": 0.80,

    "bagging_freq": 1,

    "lambda_l1": 1,

    "lambda_l2": 5,

    "feature_pre_filter": False,

    "verbosity": -1,

    "seed": 777,

    "num_threads": -1
}

In [12]:
# =============================================================================
# ЭТАП 10. Кросс-валидация ансамбля LightGBM
# =============================================================================

train_raw["_dt_parsed"] = pd.to_datetime(train_raw["dt"])

sort_idx = train_raw["_dt_parsed"].sort_values().index

X_sorted = X_train.loc[sort_idx].reset_index(drop=True)
y_sorted = y_train.loc[sort_idx].reset_index(drop=True)
w_sorted = w_train.loc[sort_idx].reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=4)

cv_scores = []

best_iters_1 = []
best_iters_2 = []
best_iters_3 = []

for fold, (train_idx, valid_idx) in enumerate(tscv.split(X_sorted), start=1):

    print("=" * 70)
    print(f"Fold {fold}")

    X_tr = X_sorted.iloc[train_idx]
    X_val = X_sorted.iloc[valid_idx]

    y_tr = y_sorted.iloc[train_idx]
    y_val = y_sorted.iloc[valid_idx]

    w_tr = w_sorted.iloc[train_idx]
    w_val = w_sorted.iloc[valid_idx]

    dtrain1 = lgb.Dataset(
    X_tr,
    label=y_tr,
    weight=w_tr,
    categorical_feature=cat_cols,
    free_raw_data=False
)

    dvalid1 = lgb.Dataset(
        X_val,
        label=y_val,
        weight=w_val,
        categorical_feature=cat_cols,
        reference=dtrain1,
        free_raw_data=False
    )

    dtrain2 = lgb.Dataset(
        X_tr,
        label=y_tr,
        weight=w_tr,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    dvalid2 = lgb.Dataset(
        X_val,
        label=y_val,
        weight=w_val,
        categorical_feature=cat_cols,
        reference=dtrain2,
        free_raw_data=False
    )

    dtrain3 = lgb.Dataset(
        X_tr,
        label=y_tr,
        weight=w_tr,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    dvalid3 = lgb.Dataset(
        X_val,
        label=y_val,
        weight=w_val,
        categorical_feature=cat_cols,
        reference=dtrain3,
        free_raw_data=False
    )

    model1 = lgb.train(
        params_1,
        dtrain1,
        valid_sets=[dvalid1],
        num_boost_round=4000,
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(0)
        ]
    )

    model2 = lgb.train(
        params_2,
        dtrain2,
        valid_sets=[dvalid2],
        num_boost_round=4000,
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(0)
        ]
    )

    model3 = lgb.train(
        params_3,
        dtrain3,
        valid_sets=[dvalid3],
        num_boost_round=4000,
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(0)
        ]
    )

    pred1 = model1.predict(
        X_val,
        num_iteration=model1.best_iteration
    )

    pred2 = model2.predict(
        X_val,
        num_iteration=model2.best_iteration
    )

    pred3 = model3.predict(
        X_val,
        num_iteration=model3.best_iteration
    )

    prediction = (
        pred1 +
        pred2 +
        pred3
    ) / 3

    score = mean_absolute_error(
        y_val,
        prediction,
        sample_weight=w_val
    )

    cv_scores.append(score)

    best_iters_1.append(model1.best_iteration)
    best_iters_2.append(model2.best_iteration)
    best_iters_3.append(model3.best_iteration)

    print(f"WMAE: {score:.2f}")

print()

print("=" * 70)
print(f"Mean WMAE : {np.mean(cv_scores):.2f}")
print(f"Std       : {np.std(cv_scores):.2f}")
print("=" * 70)

Fold 1
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1340]	valid_0's l1: 67865.5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[920]	valid_0's l1: 68329
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1991]	valid_0's l1: 67533.3
WMAE: 67643.22
Fold 2
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1798]	valid_0's l1: 64144.5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1470]	valid_0's l1: 63899.7
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[3998]	valid_0's l1: 64386.5
WMAE: 63819.96
Fold 3
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[2846]	valid_0's l1: 61610.2
Training until validation scores don't improve for 100 ro

In [22]:
print(best_iters_1)
print(best_iters_2)
print(best_iters_3)

[1340, 1798, 2846, 2390, 0]
[920, 1470, 1657, 1699, 0]
[1991, 3998, 3885, 4000, 0]


In [13]:
# =============================================================================
# ЭТАП 11. Финальное обучение ансамбля
# =============================================================================

full_train = lgb.Dataset(
    X_train,
    label=y_train,
    weight=w_train,
    categorical_feature=cat_cols
)

model1 = lgb.train(
    params_1,
    full_train,
    num_boost_round=int(np.mean(best_iters_1))
)

model2 = lgb.train(
    params_2,
    full_train,
    num_boost_round=int(np.mean(best_iters_2))
)

model3 = lgb.train(
    params_3,
    full_train,
    num_boost_round=int(np.mean(best_iters_3))
)

In [14]:
# -----------------------------------------------------------------------------
# Предсказания трех моделей
# -----------------------------------------------------------------------------

pred1 = model1.predict(
    X_val,
    num_iteration=model1.best_iteration
)

pred2 = model2.predict(
    X_val,
    num_iteration=model2.best_iteration
)

pred3 = model3.predict(
    X_val,
    num_iteration=model3.best_iteration
)

# -----------------------------------------------------------------------------
# Ансамбль
# -----------------------------------------------------------------------------

prediction = (
    0.50 * pred1 +
    0.30 * pred2 +
    0.20 * pred3
)

# -----------------------------------------------------------------------------
# Подсчет WMAE
# -----------------------------------------------------------------------------

score = mean_absolute_error(
    y_val,
    prediction,
    sample_weight=w_val
)

cv_scores.append(score)

best_iters_1.append(model1.best_iteration)
best_iters_2.append(model2.best_iteration)
best_iters_3.append(model3.best_iteration)

print(f"WMAE: {score:.2f}")
print(
    f"Best iterations: "
    f"{model1.best_iteration}, "
    f"{model2.best_iteration}, "
    f"{model3.best_iteration}"
)

WMAE: 45263.33
Best iterations: 0, 0, 0


In [15]:
print()

print("=" * 70)

print(f"Mean WMAE : {np.mean(cv_scores):.2f}")

print(f"Std WMAE  : {np.std(cv_scores):.2f}")

print(
    f"Mean best iterations: "
    f"{int(np.mean(best_iters_1))}, "
    f"{int(np.mean(best_iters_2))}, "
    f"{int(np.mean(best_iters_3))}"
)

print("=" * 70)


Mean WMAE : 59902.75
Std WMAE  : 7668.66
Mean best iterations: 1674, 1149, 2774


In [20]:
np.mean(best_iters_1)

np.float64(1674.8)

In [16]:
print(model1.best_iteration)
print(model1.current_iteration())
print(model1.num_trees())

0
2093
2093


In [19]:
best_iters_1

[1340, 1798, 2846, 2390, 0]

In [17]:
# =============================================================================
# ЭТАП 11. Финальное обучение ансамбля
# =============================================================================

full_train1 = lgb.Dataset(
    X_train,
    label=y_train,
    weight=w_train,
    categorical_feature=cat_cols,
    free_raw_data=False
)

full_train2 = lgb.Dataset(
    X_train,
    label=y_train,
    weight=w_train,
    categorical_feature=cat_cols,
    free_raw_data=False
)

full_train3 = lgb.Dataset(
    X_train,
    label=y_train,
    weight=w_train,
    categorical_feature=cat_cols,
    free_raw_data=False
)

model1 = lgb.train(
    params_1,
    full_train1,
    num_boost_round=int(np.mean(best_iters_1))
)

model2 = lgb.train(
    params_2,
    full_train2,
    num_boost_round=int(np.mean(best_iters_2))
)

model3 = lgb.train(
    params_3,
    full_train3,
    num_boost_round=int(np.mean(best_iters_3))
)

In [18]:
# =============================================================================
# ЭТАП 12. Формирование submission
# =============================================================================

pred1 = model1.predict(X_test)

pred2 = model2.predict(X_test)

pred3 = model3.predict(X_test)

prediction = (
    0.50 * pred1 +
    0.30 * pred2 +
    0.20 * pred3
)

submission = pd.DataFrame({

    "id": test_raw["id"].astype(int),

    "predict": prediction

})

submission.to_csv(
    "lgb_ensemble_v1.csv",
    sep=";",
    index=False
)

print(submission.head())

print()

print("Submission успешно сохранён.")

   id       predict
0   0  72638.705526
1   1  48781.415291
2   3  27530.505250
3   9  81941.230758
4  11  43140.171300

Submission успешно сохранён.
